In [ ]:
from datasets import load_dataset, load_from_disk
from replay.metrics import Recall, Precision, HitRate
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
import faiss
from collections import defaultdict, Counter
from functools import reduce
import datasets
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

DATA_PATH = "/home/jupyter/filestore/storage/datasets"

In [3]:
dataset = load_from_disk(f"{DATA_PATH}/user_events_20230501")
polars_ds = dataset.to_polars()

In [4]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_likes = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .filter(pl.col("event_id") == "item_like")
)

train_likes.shape

(1041535, 18)

In [5]:
sessions_with_likes = (
    train_likes
    .groupby("session_id", "user_id")
    .agg(pl.n_unique("item_id").alias("likes_cnt"))
    .filter(pl.col("likes_cnt") >= 2)
)

sessions_with_likes.shape

(179658, 3)

In [6]:
session_likes = (
    train_likes
    .join(
        sessions_with_likes,
        on=["session_id", "user_id"],
        how="inner"
    )
    .select(
        pl.struct("session_id", "user_id").apply(lambda x: f"{x['session_id']}_{x['user_id']}"),
        pl.col("item_id")
    )
    .unique()
)

In [9]:
item_occurances = defaultdict(list)

session_with_item = session_likes.groupby("item_id").agg(pl.col("session_id"))
session_items = session_likes.groupby("session_id").agg(pl.col("item_id"))
session_items = dict(zip(session_items["session_id"].to_list(), session_items["item_id"].to_list()))

item_ids = session_with_item["item_id"].to_list()
session_ids = session_with_item["session_id"].to_list()

recs_lens = []

for item_id, item_sessions in zip(item_ids, session_ids):
    for session in item_sessions:
        co_items = session_items[session]
        for item in co_items:
            if item != item_id:
                item_occurances[item_id].append(item)
    recs_lens.append(len(item_occurances[item_id]))
    
recs_lens = np.array(recs_lens)

In [10]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    dataset
    .to_polars()
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [35]:
user_last_clicks = (
     all_clicks
    .filter(pl.col("rn") <= 150)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [36]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [item_occurances[item_id] for item_id in row["last_clicks"] if item_id in item_occurances], []))

In [37]:
recs = (
    user_last_clicks
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [14]:
recs.head(5)

user_id,last_clicks,recs
i64,list[i64],list[i64]
31984352,"[36167069, 36167069, … 97908723]",[]
44312420,"[73281855, 55611300, … 184500031]","[11037938, 152088320, … 11037938]"
44589552,"[33758360, 122466753, … 100826834]","[166732797, 72866405, … 46511782]"
45827736,"[46040453, 46040453, … 46040453]",[]
47948224,"[41166007, 195835634, 171826441]","[19359788, 25754192, … 231852932]"


In [47]:
recs.rename({"recs": "co_liked_recs"}).select("user_id", "co_liked_recs").write_parquet("co_liked_recs_v2.parquet")

In [38]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs,
        on="user_id",
        how="inner"
    )
)

In [39]:
recs_stats = (
    test_interactions
    .with_columns(pl.col("recs").apply(len).alias("recs_count"))
    .select(
        pl.min("recs_count").alias("min_recs_count"),
        pl.mean("recs_count").alias("mean_recs_count"),
        pl.max("recs_count").alias("max_recs_count"),
    )
    .head(1)
)

recs_stats

min_recs_count,mean_recs_count,max_recs_count
i64,f64,i64
0,98.440202,42670


In [40]:
TOP_K_VALUES = [10, 100, 42670]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [19]:
metrics # 20 последних событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.013585,0.019869,0.020342,0.017535,0.002869,0.000029,0.112826,0.152511,0.154932,13098.0,17705.0,17986.0


In [27]:
metrics # 40 последних событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.013009,0.022765,0.024307,0.017368,0.003631,0.000038,0.1128,0.180059,0.188388,13095.0,20903.0,21870.0


In [34]:
metrics # 80 последних событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.011773,0.023111,0.026977,0.015122,0.003932,0.000025,0.10037,0.18794,0.208881,11652.0,21818.0,24249.0


In [42]:
metrics # 150 последних событий

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.011232,0.022448,0.028283,0.013266,0.003745,0.000014,0.091326,0.183452,0.217443,10602.0,21297.0,25243.0
